In [ ]:
!pip install xgboost

In [1]:
import pandas as pd
import numpy as np

# 1. 读取常规赛详细数据
df_reg = pd.read_csv("MRegularSeasonDetailedResults.csv")

# ==========================================
# 2. 核心特征工程：提取高阶特征 (Four Factors)
# 赢球方和输球方的数据要分别整理，然后再拼接到一起
# ==========================================

# -- 处理赢球方的数据 --
win_stats = df_reg[['Season', 'WTeamID', 'WScore', 'LScore', 'WFGM', 'WFGA', 'WFGM3', 'WFTA', 'WTO', 'WOR', 'LDR']].copy()
win_stats['Wins'] = 1
win_stats['Games'] = 1
# 计算有效命中率分子
win_stats['eFG_numerator'] = win_stats['WFGM'] + 0.5 * win_stats['WFGM3']
# 统一列名，方便后续合并
win_stats.columns = ['Season', 'TeamID', 'PointsScored', 'PointsAllowed', 'FGM', 'FGA', 'FGM3', 'FTA', 'TO', 'OR', 'OppDR', 'Wins', 'Games', 'eFG_numerator']

# -- 处理输球方的数据 --
lose_stats = df_reg[['Season', 'LTeamID', 'LScore', 'WScore', 'LFGM', 'LFGA', 'LFGM3', 'LFTA', 'LTO', 'LOR', 'WDR']].copy()
lose_stats['Wins'] = 0
lose_stats['Games'] = 1
# 计算有效命中率分子
lose_stats['eFG_numerator'] = lose_stats['LFGM'] + 0.5 * lose_stats['LFGM3']
# 统一列名
lose_stats.columns = ['Season', 'TeamID', 'PointsScored', 'PointsAllowed', 'FGM', 'FGA', 'FGM3', 'FTA', 'TO', 'OR', 'OppDR', 'Wins', 'Games', 'eFG_numerator']

# -- 将赢球和输球的数据上下拼接到一起 --
all_games = pd.concat([win_stats, lose_stats], ignore_index=True)

# ==========================================
# 3. 按赛季 (Season) 和 球队 (TeamID) 分组聚合汇总
# ==========================================
team_season_stats = all_games.groupby(['Season', 'TeamID']).sum().reset_index()

# ==========================================
# 4. 计算终极高阶指标 (Advanced Metrics)
# ==========================================
# 胜率
team_season_stats['WinRate'] = team_season_stats['Wins'] / team_season_stats['Games']
# 场均净胜分 (极其重要的特征)
team_season_stats['AvgScoreMargin'] = (team_season_stats['PointsScored'] - team_season_stats['PointsAllowed']) / team_season_stats['Games']
# 有效命中率 (eFG%)
team_season_stats['eFG%'] = team_season_stats['eFG_numerator'] / team_season_stats['FGA']
# 失误率 (TOV%) - 估算公式
team_season_stats['TOV%'] = team_season_stats['TO'] / (team_season_stats['FGA'] + 0.44 * team_season_stats['FTA'] + team_season_stats['TO'])
# 进攻篮板率 (ORB%)
team_season_stats['ORB%'] = team_season_stats['OR'] / (team_season_stats['OR'] + team_season_stats['OppDR'])

# 5. 只保留我们需要喂给模型的精华特征
final_team_features = team_season_stats[['Season', 'TeamID', 'WinRate', 'AvgScoreMargin', 'eFG%', 'TOV%', 'ORB%']]

# 打印结果看看
print(final_team_features.head(10))
# final_team_features.to_csv("Team_Season_Features.csv", index=False) # 运行这行可以保存结果

   Season  TeamID   WinRate  AvgScoreMargin      eFG%      TOV%      ORB%
0    2003    1102  0.428571        0.250000  0.579443  0.194557  0.171806
1    2003    1103  0.481481        0.629630  0.534814  0.158154  0.307334
2    2003    1104  0.607143        4.285714  0.475953  0.166753  0.374753
3    2003    1105  0.269231       -4.884615  0.457241  0.207538  0.338476
4    2003    1106  0.464286       -0.142857  0.479005  0.214109  0.354639
5    2003    1107  0.250000       -9.785714  0.487259  0.165432  0.233569
6    2003    1108  0.424242       -4.030303  0.469040  0.213606  0.356086
7    2003    1110  0.533333        4.466667  0.514633  0.177989  0.301811
8    2003    1111  0.615385        1.538462  0.499404  0.175589  0.328155
9    2003    1112  0.892857       14.964286  0.514946  0.161593  0.394615


In [5]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score

# ==========================================
# 1. 准备“真题”：读取锦标赛历史赛果
# ==========================================
df_tourney = pd.read_csv("MNCAATourneyDetailedResults.csv")
df_tourney = df_tourney[['Season', 'WTeamID', 'LTeamID']]

# 构建对称对阵表 (防止模型产生“永远选左边”的偏见)
df_win = df_tourney.copy()
df_win.rename(columns={'WTeamID': 'TeamA', 'LTeamID': 'TeamB'}, inplace=True)
df_win['Target'] = 1

df_lose = df_tourney.copy()
df_lose.rename(columns={'LTeamID': 'TeamA', 'WTeamID': 'TeamB'}, inplace=True)
df_lose['Target'] = 0

df_matchups = pd.concat([df_win, df_lose], ignore_index=True)

# ==========================================
# 2. 【核心魔法】把我们算好的特征拼接到对阵表里
# ==========================================
# 提取 TeamA 的特征
df_model = pd.merge(df_matchups, final_team_features, 
                    left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left')
df_model.rename(columns={'WinRate': 'WinRateA', 'AvgScoreMargin': 'MarginA', 
                         'eFG%': 'eFG_A', 'TOV%': 'TOV_A', 'ORB%': 'ORB_A'}, inplace=True)
df_model.drop('TeamID', axis=1, inplace=True)

# 提取 TeamB 的特征
df_model = pd.merge(df_model, final_team_features, 
                    left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left')
df_model.rename(columns={'WinRate': 'WinRateB', 'AvgScoreMargin': 'MarginB', 
                         'eFG%': 'eFG_B', 'TOV%': 'TOV_B', 'ORB%': 'ORB_B'}, inplace=True)
df_model.drop('TeamID', axis=1, inplace=True)

# 只要特征做差 (TeamA - TeamB)！机器最喜欢看差距
df_model['Diff_WinRate'] = df_model['WinRateA'] - df_model['WinRateB']
df_model['Diff_Margin'] = df_model['MarginA'] - df_model['MarginB']
df_model['Diff_eFG'] = df_model['eFG_A'] - df_model['eFG_B']
df_model['Diff_TOV'] = df_model['TOV_A'] - df_model['TOV_B']
df_model['Diff_ORB'] = df_model['ORB_A'] - df_model['ORB_B']

# ==========================================
# 3. 开始训练 XGBoost 模型！
# ==========================================
# 挑选特征列作为 X
features = ['Diff_WinRate', 'Diff_Margin', 'Diff_eFG', 'Diff_TOV', 'Diff_ORB']
X = df_model[features]
y = df_model['Target']

# 切分训练集和验证集 (80% 训练，20% 考试)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# 定义 XGBoost 模型 (这里可以自己调参哦)
model = XGBClassifier(n_estimators=100, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

# 预测概率 (因为 Kaggle 比赛要求提交的是赢球的概率)
y_pred_proba = model.predict_proba(X_valid)[:, 1]
y_pred_class = model.predict(X_valid)

# 评估分数
print("============== 训练完成！==============")
print(f"模型准确率 (Accuracy): {accuracy_score(y_valid, y_pred_class):.4f}")
print(f"Brier Score / Log Loss (Kaggle 评分标准): {log_loss(y_valid, y_pred_proba):.4f}")

# 查看哪个特征最重要 (Feature Importance)
importance = pd.DataFrame({'Feature': features, 'Importance': model.feature_importances_})
print("\n特征重要性排名：")
print(importance.sort_values(by='Importance', ascending=False))

============== 训练完成！==============
模型准确率 (Accuracy): 0.6707
Brier Score / Log Loss (Kaggle 评分标准): 0.5774

特征重要性排名：
        Feature  Importance
1   Diff_Margin    0.492165
0  Diff_WinRate    0.140789
3      Diff_TOV    0.127081
4      Diff_ORB    0.122922
2      Diff_eFG    0.117044


In [6]:
# ==========================================
# 1. 读取并清洗种子数据
# ==========================================
df_seeds = pd.read_csv("MNCAATourneySeeds.csv")

# 魔法清洗：把 "W01" 变成数字 1，把 "Z16a" 变成数字 16
# 这段代码会自动遍历 Seed 列，只保留里面的数字部分并转成整数
df_seeds['SeedNum'] = df_seeds['Seed'].apply(lambda x: int(''.join([c for c in x if c.isdigit()])))

# 我们只需要 赛季、球队ID、和干净的数字种子
df_seeds = df_seeds[['Season', 'TeamID', 'SeedNum']]

print("1. 清洗后的种子表长这样：")
display(df_seeds.head())

# ==========================================
# 2. 把“种子顺位”融合进我们之前的大表 (df_model) 中
# ==========================================
# 注意：这里假设你之前跑的 df_model 还在内存里哦！

# 给 Team A 贴上种子标签
df_model_v2 = pd.merge(df_model, df_seeds, 
                       left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left')
df_model_v2.rename(columns={'SeedNum': 'SeedA'}, inplace=True)
df_model_v2.drop('TeamID', axis=1, inplace=True)

# 给 Team B 贴上种子标签
df_model_v2 = pd.merge(df_model_v2, df_seeds, 
                       left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left')
df_model_v2.rename(columns={'SeedNum': 'SeedB'}, inplace=True)
df_model_v2.drop('TeamID', axis=1, inplace=True)

# 创造极其致命的新特征：【种子顺位差】
df_model_v2['Diff_Seed'] = df_model_v2['SeedA'] - df_model_v2['SeedB']

# 极少数早期比赛可能没记录种子，防止报错，用 0 填补空缺（代表实力相当）
df_model_v2.fillna(0, inplace=True)


# ==========================================
# 3. 带着新特征，重新训练 XGBoost 2.0！
# ==========================================
# 现在的特征池里多了一位重量级选手：'Diff_Seed'
features_v2 = ['Diff_WinRate', 'Diff_Margin', 'Diff_eFG', 'Diff_TOV', 'Diff_ORB', 'Diff_Seed']
X_v2 = df_model_v2[features_v2]
y_v2 = df_model_v2['Target']

# 切分数据
X_train_v2, X_valid_v2, y_train_v2, y_valid_v2 = train_test_split(X_v2, y_v2, test_size=0.2, random_state=42)

# 训练升级版模型
model_v2 = XGBClassifier(n_estimators=100, learning_rate=0.05, random_state=42)
model_v2.fit(X_train_v2, y_train_v2)

# 预测并出分
y_pred_proba_v2 = model_v2.predict_proba(X_valid_v2)[:, 1]
y_pred_class_v2 = model_v2.predict(X_valid_v2)

print("\n============== 👑 升级版模型训练完成！==============")
print(f"新模型准确率 (Accuracy): {accuracy_score(y_valid_v2, y_pred_class_v2):.4f}")
print(f"新 Brier Score / Log Loss: {log_loss(y_valid_v2, y_pred_proba_v2):.4f}")

# 查看新特征是不是真的那么强？
importance_v2 = pd.DataFrame({'Feature': features_v2, 'Importance': model_v2.feature_importances_})
print("\n新特征重要性排名：")
print(importance_v2.sort_values(by='Importance', ascending=False))

1. 清洗后的种子表长这样：


,Season,TeamID,SeedNum
0,1985,1207,1
1,1985,1210,2
2,1985,1228,3
3,1985,1260,4
4,1985,1374,5



============== 👑 升级版模型训练完成！==============
新模型准确率 (Accuracy): 0.7017
新 Brier Score / Log Loss: 0.5883

新特征重要性排名：
        Feature  Importance
5     Diff_Seed    0.628348
1   Diff_Margin    0.086689
0  Diff_WinRate    0.079378
3      Diff_TOV    0.074552
4      Diff_ORB    0.068908
2      Diff_eFG    0.062125


In [7]:
from sklearn.model_selection import GridSearchCV

# ==========================================
# 专治过度自信的 XGBoost 3.0 (调参版)
# ==========================================
model_tuned = XGBClassifier(
    n_estimators=200,        # 树的数量变多一点，博采众长
    learning_rate=0.01,      # 【紧箍咒 1】学习率调低(原来0.05)。步子迈小点，学得稳一点
    max_depth=3,             # 【紧箍咒 2】最大深度(默认是6)。改成3，不让它想得太复杂，防止钻牛角尖
    min_child_weight=5,      # 【紧箍咒 3】防止极端概率。要求每个结论至少要有5个历史相似样本支撑
    subsample=0.8,           # 【紧箍咒 4】每次建树只随机抽取80%的比赛，防止死记硬背
    colsample_bytree=0.8,    # 【紧箍咒 5】每次建树只随机看80%的特征，逼迫它不能每次都只靠 Diff_Seed 偷懒
    random_state=42
)

# 重新训练！(直接用我们刚才切分好的 v2 版本的数据)
model_tuned.fit(X_train_v2, y_train_v2)

# 预测概率
y_pred_proba_tuned = model_tuned.predict_proba(X_valid_v2)[:, 1]
y_pred_class_tuned = model_tuned.predict(X_valid_v2)

print("\n============== 🧘‍♂️ 调参完毕：谦虚版模型 ==============")
print(f"调参后准确率 (Accuracy): {accuracy_score(y_valid_v2, y_pred_class_tuned):.4f}")
print(f"调参后 Brier Score / Log Loss: {log_loss(y_valid_v2, y_pred_proba_tuned):.4f}")

# 对比一下之前的最高概率和最低概率
print("\n🔍 看看模型是不是变得谦虚了：")
print(f"旧模型给出的最高赢球概率: {max(y_pred_proba_v2):.4f}")
print(f"新模型给出的最高赢球概率: {max(y_pred_proba_tuned):.4f}")


============== 🧘‍♂️ 调参完毕：谦虚版模型 ==============
调参后准确率 (Accuracy): 0.7138
调参后 Brier Score / Log Loss: 0.5648

🔍 看看模型是不是变得谦虚了：
旧模型给出的最高赢球概率: 0.9911
新模型给出的最高赢球概率: 0.8996


In [10]:
# ==========================================
# 1. 读取并过滤 128MB 的排名数据
# ==========================================
print("正在读取 128MB 的排名数据，请稍等...")
df_rankings = pd.read_csv("MMasseyOrdinals.csv")

# 【核心过滤】只取锦标赛开打前最后一天 (DayNum=133) 的 KenPom (POM) 排名
# KenPom 综合了进攻效率、防守效率和赛程难度，是全美公认最权威的预测榜单
df_pom = df_rankings[(df_rankings['RankingDayNum'] == 133) & (df_rankings['SystemName'] == 'POM')].copy()
# 我们只需要 赛季、球队ID、和 排名(OrdinalRank)
df_pom = df_pom[['Season', 'TeamID', 'OrdinalRank']]

print("1. 提纯后的权威排名表长这样：")
display(df_pom.head())

# ==========================================
# 2. 把排名融合进我们之前的 v2 数据表 (df_model_v2) 中
# ==========================================
# 复制一份之前的数据，保护现场
df_model_v3 = df_model_v2.copy()

# 给 Team A 贴上排名
df_model_v3 = pd.merge(df_model_v3, df_pom, 
                       left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left')
df_model_v3.rename(columns={'OrdinalRank': 'RankA'}, inplace=True)
df_model_v3.drop('TeamID', axis=1, inplace=True)

# 给 Team B 贴上排名
df_model_v3 = pd.merge(df_model_v3, df_pom, 
                       left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left')
df_model_v3.rename(columns={'OrdinalRank': 'RankB'}, inplace=True)
df_model_v3.drop('TeamID', axis=1, inplace=True)

# 创造终极新特征：【排名差】 
# (注意：排名数字越小越强，比如第1名比第100名强。如果 A是第1，B是第10，差值是 -9，负数说明 A 强)
df_model_v3['Diff_Rank'] = df_model_v3['RankA'] - df_model_v3['RankB']

# 填补极少数的空缺值 (2003年之前的早期比赛可能没有 POM 排名，用 0 填补)
df_model_v3.fillna(0, inplace=True)

# ==========================================
# 3. 带着 7 大神级特征，训练 XGBoost 3.0！
# ==========================================
features_v3 = ['Diff_WinRate', 'Diff_Margin', 'Diff_eFG', 'Diff_TOV', 'Diff_ORB', 'Diff_Seed', 'Diff_Rank']
X_v3 = df_model_v3[features_v3]
y_v3 = df_model_v3['Target']

# 切分数据
X_train_v3, X_valid_v3, y_train_v3, y_valid_v3 = train_test_split(X_v3, y_v3, test_size=0.2, random_state=42)

# 继续用我们刚才调好的“谦虚版”参数 (专治过度自信)
model_v3 = XGBClassifier(
    n_estimators=200,
    learning_rate=0.01,
    max_depth=3,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model_v3.fit(X_train_v3, y_train_v3)

# 预测并出分
y_pred_proba_v3 = model_v3.predict_proba(X_valid_v3)[:, 1]
y_pred_class_v3 = model_v3.predict(X_valid_v3)

print("\n============== 🔮 融合媒体排名：终极版模型 ==============")
print(f"终极版准确率 (Accuracy): {accuracy_score(y_valid_v3, y_pred_class_v3):.4f}")
print(f"终极版 Brier Score / Log Loss: {log_loss(y_valid_v3, y_pred_proba_v3):.4f}")

# 查看新特征是不是真的那么强？
importance_v3 = pd.DataFrame({'Feature': features_v3, 'Importance': model_v3.feature_importances_})
print("\n终极特征重要性排名：")
print(importance_v3.sort_values(by='Importance', ascending=False))

正在读取 128MB 的排名数据，请稍等...
1. 提纯后的权威排名表长这样：


,Season,TeamID,OrdinalRank
127259,2003,1102,160
127260,2003,1103,163
127261,2003,1104,33
127262,2003,1105,307
127263,2003,1106,263



============== 🔮 融合媒体排名：终极版模型 ==============
终极版准确率 (Accuracy): 0.7207
终极版 Brier Score / Log Loss: 0.5612

终极特征重要性排名：
        Feature  Importance
6     Diff_Rank    0.442948
5     Diff_Seed    0.254354
1   Diff_Margin    0.109441
0  Diff_WinRate    0.064246
3      Diff_TOV    0.048273
4      Diff_ORB    0.043631
2      Diff_eFG    0.037108


In [13]:
import pandas as pd
import numpy as np

# ==========================================
# 1. 读取官方给的“答题卡” 
# (⚠️ 请把这里的文件名改成你电脑里实际的官方提交模板名)
# ==========================================
submission_file = "SampleSubmissionStage1.csv"  # 可能是 SampleSubmissionStage1.csv 等
df_sub = pd.read_csv(submission_file)

# 拆解 ID，把 赛季 和 对阵双方 剥离出来
# ID 的格式是 "Season_TeamA_TeamB"
df_sub['Season'] = df_sub['ID'].apply(lambda x: int(x.split('_')[0]))
df_sub['TeamA']  = df_sub['ID'].apply(lambda x: int(x.split('_')[1]))
df_sub['TeamB']  = df_sub['ID'].apply(lambda x: int(x.split('_')[2]))

print("1. 正在准备 7 大终极特征...")

# ==========================================
# 2. 贴上我们所有的终极特征！(大合体)
# ==========================================
# 【贴常规赛战力】
df_sub = pd.merge(df_sub, final_team_features, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left')
df_sub.rename(columns={'WinRate': 'WinRateA', 'AvgScoreMargin': 'MarginA', 'eFG%': 'eFG_A', 'TOV%': 'TOV_A', 'ORB%': 'ORB_A'}, inplace=True)
df_sub.drop('TeamID', axis=1, inplace=True)

df_sub = pd.merge(df_sub, final_team_features, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left')
df_sub.rename(columns={'WinRate': 'WinRateB', 'AvgScoreMargin': 'MarginB', 'eFG%': 'eFG_B', 'TOV%': 'TOV_B', 'ORB%': 'ORB_B'}, inplace=True)
df_sub.drop('TeamID', axis=1, inplace=True)

# 【贴种子顺位】
df_sub = pd.merge(df_sub, df_seeds, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left')
df_sub.rename(columns={'SeedNum': 'SeedA'}, inplace=True)
df_sub.drop('TeamID', axis=1, inplace=True)

df_sub = pd.merge(df_sub, df_seeds, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left')
df_sub.rename(columns={'SeedNum': 'SeedB'}, inplace=True)
df_sub.drop('TeamID', axis=1, inplace=True)

# 【贴媒体排名 POM】
df_sub = pd.merge(df_sub, df_pom, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left')
df_sub.rename(columns={'OrdinalRank': 'RankA'}, inplace=True)
df_sub.drop('TeamID', axis=1, inplace=True)

df_sub = pd.merge(df_sub, df_pom, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left')
df_sub.rename(columns={'OrdinalRank': 'RankB'}, inplace=True)
df_sub.drop('TeamID', axis=1, inplace=True)

# 填补极少数空缺值 (如果没有种子或排名，一律用 0 或极弱的排位填补)
df_sub.fillna(0, inplace=True)

# ==========================================
# 3. 计算特征差值 (只做减法！)
# ==========================================
df_sub['Diff_WinRate'] = df_sub['WinRateA'] - df_sub['WinRateB']
df_sub['Diff_Margin']  = df_sub['MarginA']  - df_sub['MarginB']
df_sub['Diff_eFG']     = df_sub['eFG_A']    - df_sub['eFG_B']
df_sub['Diff_TOV']     = df_sub['TOV_A']    - df_sub['TOV_B']
df_sub['Diff_ORB']     = df_sub['ORB_A']    - df_sub['ORB_B']
df_sub['Diff_Seed']    = df_sub['SeedA']    - df_sub['SeedB']
df_sub['Diff_Rank']    = df_sub['RankA']    - df_sub['RankB']

# ==========================================
# 4. 让终极模型作答！
# ==========================================
# 提取出跟训练时一模一样的 7 个特征列 (注意顺序必须和模型要求的一致)
X_sub = df_sub[['Diff_WinRate', 'Diff_Margin', 'Diff_eFG', 'Diff_TOV', 'Diff_ORB', 'Diff_Seed', 'Diff_Rank']]

print("2. 终极模型正在预测今年所有的对阵结果...")
# 使用调优过的 3.0 模型预测
df_sub['Pred'] = model_v3.predict_proba(X_sub)[:, 1]

# ==========================================
# 5. 生成最终提交文件
# ==========================================
# 丢掉所有中间计算的列，只保留 Kaggle 要求的 ID 和 Pred
final_submission = df_sub[['ID', 'Pred']]

# 保存为 CSV 文件
final_submission.to_csv("Master_Submission_V3.csv", index=False)

print("\n🏆 大功告成！最终预测文件 'Master_Submission_V3.csv' 已经生成在你的本地文件夹里了！")
display(final_submission.head())


1. 正在准备 7 大终极特征...
2. 终极模型正在预测今年所有的对阵结果...

🏆 大功告成！最终预测文件 'Master_Submission_V3.csv' 已经生成在你的本地文件夹里了！


,ID,Pred
0,2022_1101_1102,0.784332
1,2022_1101_1103,0.686096
2,2022_1101_1104,0.352020
3,2022_1101_1105,0.766009
4,2022_1101_1106,0.776876
